# ECCOFS `avg`: surface temperature from the best-estimate virtual Icechunk store

The store at `https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best` on [Source Cooperative](https://source.coop) is a virtual Icechunk repository over the NOAA ECCOFS `avg` NetCDF files.
It already holds the best-estimate series: one entry per valid time, a single monotonic `time` dimension, so no forecast bookkeeping (and no Rolodex) is needed.
Each time comes from the `_0002` file of its run (the middle day of the 3-day analysis); the last six times come from the rest of the latest run.

This notebook:
1. opens the store from Source Cooperative,
2. picks the surface level of `temp`,
3. plots a map and a point time series with hvPlot.

No credentials are needed: both Source Cooperative (which holds the store) and NOAA's bucket (which holds the NetCDF bytes) are read anonymously. It needs icechunk 2.x, which needs Python 3.12 or newer.

In [ ]:
import hvplot.xarray  # noqa: F401
import icechunk
import xarray as xr
import zarr

zarr.config.set({"async.concurrency": 32})

URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

In [ ]:
repo = icechunk.Repository.open(icechunk.http_storage(URL))
# the NetCDF bytes live in NOAA's bucket, outside the store: authorize that virtual chunk container (anonymous)
repo = repo.reopen(authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}))
latest = next(iter(repo.ancestry(branch="main")))
print(latest.written_at, "|", latest.message)

ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
t = ds.indexes["time"]
assert t.is_monotonic_increasing and t.is_unique
print(f"{len(t)} times: {t[0]} to {t[-1]}")
ds[["temp"]]

## Surface layer of `temp`

ROMS orders `s_rho` from the bottom (-1) to the surface (0), so the surface is the last level. The assertion guards against that order being reversed. One level is one chunk, so only the surface level is read.

In [ ]:
assert float(ds.s_rho[-1]) > float(ds.s_rho[0]), "expected s_rho to increase towards the surface"
sst = ds.temp.isel(s_rho=-1).rename("sst")
sst.attrs.update(long_name="surface potential temperature", units="degC")
sst

## Map

The grid is curvilinear (`lon_rho`, `lat_rho`), so this uses `quadmesh` with datashader rasterization. It reads one surface level from NOAA's public bucket.

In [ ]:
%%time
sst_map = sst.sel(time="2026-11-01 12:00").load()

In [ ]:
sst_map.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, tiles="OSM",
    cmap="turbo", geo=True, coastline=True, frame_width=700,
    title="ECCOFS best-estimate surface temperature (degC), 2026-11-01",
)

## Point time series

The nearest grid cell to a location, over every time in the store (one small chunk per time).

In [ ]:
%%time
import numpy as np
lon, lat = ds.lon_rho.load(), ds.lat_rho.load()
LON, LAT = -70.0, 41.0   # edit me
j, i = np.unravel_index(np.nanargmin(((lon - LON) ** 2 + (lat - LAT) ** 2).values), lon.shape)
point = sst.isel(eta_rho=j, xi_rho=i).load()
print(f"nearest cell: lon={float(lon[j, i]):.3f}, lat={float(lat[j, i]):.3f}")

In [ ]:
point.hvplot.line(x="time", ylabel="degC", title=f"Surface temperature near ({LON}, {LAT})", grid=True)